In [ ]:
import os
import html
import webbrowser
import folium
import pandas as pd
from folium.plugins import MarkerCluster, FastMarkerCluster

# ---------- CHEMINS ----------
dossier = '/data/data/'
dossier_ref = '/ref/ref/'

def lire(nom, base=dossier, cols=None):
    """Lit un fichier ANFR (séparateur ';'), colonnes normalisées en MAJUSCULES.
    cols = liste des colonnes à garder (plus rapide et moins de mémoire)."""
    chemin = base + nom
    if not os.path.exists(chemin):
        raise FileNotFoundError(f"Fichier introuvable : {chemin}")
    filtre = (lambda c: c.strip().upper() in cols) if cols else None
    try:
        d = pd.read_csv(chemin, sep=";", dtype=str, encoding="utf-8-sig", usecols=filtre)
    except UnicodeDecodeError:
        d = pd.read_csv(chemin, sep=";", dtype=str, encoding="latin-1", usecols=filtre)
    d.columns = d.columns.str.strip().str.upper()
    return d

def joindre(serie):
    """Valeurs uniques, triées, séparées par des virgules."""
    return ', '.join(sorted(set(serie.dropna().astype(str))))

# ---------- 1. SUPPORTS ----------
sup = lire('sup_support.txt')

# Conversion des coordonnées (degrés, minutes, secondes) en degrés décimaux
for c in ['COR_NB_DG_LAT', 'COR_NB_MN_LAT', 'COR_NB_SC_LAT',
          'COR_NB_DG_LON', 'COR_NB_MN_LON', 'COR_NB_SC_LON']:
    sup[c] = pd.to_numeric(sup[c], errors='coerce')

sup['lat'] = sup['COR_NB_DG_LAT'] + sup['COR_NB_MN_LAT'] / 60 + sup['COR_NB_SC_LAT'] / 3600
sup.loc[sup['COR_CD_NS_LAT'] == 'S', 'lat'] *= -1          # Sud = négatif
sup['lon'] = sup['COR_NB_DG_LON'] + sup['COR_NB_MN_LON'] / 60 + sup['COR_NB_SC_LON'] / 3600
sup.loc[sup['COR_CD_EW_LON'] == 'W', 'lon'] *= -1          # Ouest = négatif
sup = sup.dropna(subset=['lat', 'lon'])

# Filtre département (avant les jointures : ça les rend beaucoup plus rapides)
dep = input("Département (vide = toute la France) : ").strip().upper()
if dep:
    dep = dep.zfill(2)     # "1" devient "01"
    sup = sup[sup['ADR_NM_CP'].str.startswith(dep, na=False)]

# On garde le lien support <-> installation AVANT de supprimer les doublons,
# car un support peut avoir plusieurs installations (plusieurs opérateurs)
liens = sup[['SUP_ID', 'STA_NM_ANFR']].drop_duplicates()
ids_sup = set(sup['SUP_ID'])
ids_sta = set(liens['STA_NM_ANFR'])
sup = sup.drop_duplicates(subset='SUP_ID')

# ---------- 2. NATURE + PROPRIÉTAIRE (tables de référence) ----------
sup = sup.merge(lire('sup_nature.txt', dossier_ref), on='NAT_ID', how='left')
sup = sup.merge(lire('sup_proprietaire.txt', dossier_ref), on='TPO_ID', how='left')

# ---------- 3. EXPLOITANTS + DATE DE MISE EN SERVICE ----------
station = lire('sup_station.txt')
station = station[station['STA_NM_ANFR'].isin(ids_sta)]
station = station.merge(lire('sup_exploitant.txt', dossier_ref), on='ADM_ID', how='left')
station['DATE'] = pd.to_datetime(station['DTE_EN_SERVICE'], dayfirst=True, errors='coerce')
station = liens.merge(station[['STA_NM_ANFR', 'ADM_LB_NOM', 'DATE']],
                      on='STA_NM_ANFR', how='left')
infos_station = station.groupby('SUP_ID').agg(
    exploitants=('ADM_LB_NOM', joindre),
    mise_service=('DATE', 'min')
).reset_index()

# ---------- 4. ANTENNES (types, nombre) ----------
ant = lire('sup_antenne.txt', cols=['STA_NM_ANFR', 'AER_ID', 'TAE_ID', 'SUP_ID'])
ant = ant[ant['SUP_ID'].isin(ids_sup)]
ant_t = ant.merge(lire('sup_type_antenne.txt', dossier_ref), on='TAE_ID', how='left')
infos_ant = ant_t.groupby('SUP_ID').agg(
    nb_antennes=('AER_ID', 'nunique'),
    types_antenne=('TAE_LB', joindre)
).reset_index()

# ---------- 5. ÉMETTEURS (systèmes : GSM, LTE, 5G...) ----------
emr = lire('sup_emetteur.txt', cols=['STA_NM_ANFR', 'AER_ID', 'EMR_LB_SYSTEME'])
emr = emr[emr['STA_NM_ANFR'].isin(ids_sta)]
emr = emr.merge(ant[['STA_NM_ANFR', 'AER_ID', 'SUP_ID']],
                on=['STA_NM_ANFR', 'AER_ID'], how='inner')
infos_emr = emr.groupby('SUP_ID').agg(systemes=('EMR_LB_SYSTEME', joindre)).reset_index()

# ---------- 6. ASSEMBLAGE ----------
df = (sup.merge(infos_station, on='SUP_ID', how='left')
         .merge(infos_ant, on='SUP_ID', how='left')
         .merge(infos_emr, on='SUP_ID', how='left'))

def txt(v, defaut='?'):
    """Texte sécurisé pour le HTML, ou valeur par défaut si vide."""
    return html.escape(str(v)) if pd.notna(v) and str(v).strip() else defaut

def popup(r):
    adresse = r.ADR_LB_LIEU if pd.notna(r.ADR_LB_LIEU) else r.ADR_LB_ADD1
    date = r.mise_service.strftime('%d/%m/%Y') if pd.notna(r.mise_service) else '?'
    return (
        f"<b>{txt(adresse, 'Support')}</b><br>"
        f"{txt(r.ADR_NM_CP, '')}<hr style='margin:4px 0'>"
        f"<b>Nature :</b> {txt(r.NAT_LB_NOM)}<br>"
        f"<b>Hauteur :</b> {txt(r.SUP_NM_HAUT)} m<br>"
        f"<b>Propriétaire :</b> {txt(r.TPO_LB)}<br>"
        f"<b>Exploitant(s) :</b> {txt(r.exploitants)}<br>"
        f"<b>Mise en service :</b> {date}<br>"
        f"<b>Antennes :</b> {txt(r.nb_antennes, '0')} ({txt(r.types_antenne)})<br>"
        f"<b>Systèmes :</b> {txt(r.systemes)}"
    )

df['popup'] = [popup(r) for r in df.itertuples()]

# ---------- 7. CARTE ----------
Carte = folium.Map(location=[47, 1], zoom_start=6, tiles=None, min_zoom=4,
                   prefer_canvas=True)

folium.TileLayer('CartoDB positron', name='CartoDB clair', no_wrap=True).add_to(Carte)
folium.TileLayer('CartoDB voyager', name='CartoDB couleur', no_wrap=True, show=False).add_to(Carte)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/'
          'World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles © Esri', name='Esri World Street Map', no_wrap=True, show=False,
).add_to(Carte)

print(f"{len(df)} supports affichés")

if len(df) == 0:
    print("Aucun support : vérifie le département (ex. 76, 27, 2A).")
elif len(df) <= 15000:
    # Mode normal : ronds cliquables regroupés en paquets
    groupe = MarkerCluster(name='Supports').add_to(Carte)
    for r in df.itertuples():
        folium.CircleMarker(
            location=[r.lat, r.lon],
            radius=6,
            color='#c0392b', fill=True, fill_color='#e74c3c', fill_opacity=0.8,
            popup=folium.Popup(r.popup, max_width=350),
        ).add_to(groupe)
else:
    # Mode rapide pour de très gros volumes
    donnees = df[['lat', 'lon', 'popup']].values.tolist()
    callback = """
    function (row) {
        var marker = L.circleMarker([row[0], row[1]], {radius: 5, color: '#c0392b'});
        marker.bindPopup(row[2], {maxWidth: 350});
        return marker;
    };
    """
    FastMarkerCluster(donnees, callback=callback).add_to(Carte)

# Recentre la carte sur les supports affichés
if len(df) > 0:
    Carte.fit_bounds([[df['lat'].min(), df['lon'].min()],
                      [df['lat'].max(), df['lon'].max()]])

folium.LayerControl().add_to(Carte)
Carte.save('carte_code05.html')
webbrowser.open('file://' + os.path.abspath('carte_code05.html'))

FileNotFoundError: Fichier introuvable : /SAE15_C2/data/data/sup_support.txt